# 🧠 Module 5.1: Neural Network Basics

## From a Single Neuron to Deep Networks — The Complete Mathematical Foundation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_05_Deep_Learning_For_Images/01_Neural_Network_Basics/neural_network_basics.ipynb)

---

## 🎯 Learning Objectives

1. Understand the perceptron model and its geometric interpretation
2. Master activation functions (ReLU, Sigmoid, Tanh) and their derivatives
3. Derive the forward pass mathematically for multi-layer networks
4. Perform full backpropagation derivation using the chain rule
5. Implement gradient descent from scratch
6. Build a complete neural network in NumPy to classify tiny images

---

In [ ]:
# Google Colab Setup
!pip install numpy matplotlib scikit-learn -q

In [ ]:
# Download REAL open-source datasets (TINY - under 250MB total)
import torchvision
import torchvision.transforms as transforms

transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])

# MNIST (handwritten digits - 11MB)
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# CIFAR-10 (real photographs - 170MB)
transform_cifar = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))])
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_cifar)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_cifar)

# FashionMNIST (clothing items - 30MB, great for transfer learning!)
fashion_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
fashion_test = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

print(f"✅ MNIST: {len(mnist_train)} train + {len(mnist_test)} test (28x28 grayscale)")
print(f"✅ CIFAR-10: {len(cifar_train)} train + {len(cifar_test)} test (32x32 RGB)")
print(f"✅ FashionMNIST: {len(fashion_train)} train + {len(fashion_test)} test (28x28)")
print(f"   Classes: {cifar_train.classes}")
print(f"\n📦 Total download: ~211MB (NO STL-10 needed!)")

## Deep Derivation: Backpropagation from Chain Rule

### Step 1: Forward Pass (Single Neuron)
$$z = \sum_{i=1}^n w_i x_i + b = \mathbf{w}^T\mathbf{x} + b$$
$$a = \sigma(z)$$

### Step 2: Loss Function (MSE for one sample)
$$L = \frac{1}{2}(y - a)^2$$

### Step 3: Gradient of Loss w.r.t. Output
$$\frac{\partial L}{\partial a} = -(y - a) = a - y$$

### Step 4: Gradient Through Activation (Sigmoid)
$$\sigma(z) = \frac{1}{1+e^{-z}}, \quad \sigma'(z) = \sigma(z)(1-\sigma(z))$$

**Proof of sigmoid derivative:**
$$\frac{d\sigma}{dz} = \frac{d}{dz}\left(\frac{1}{1+e^{-z}}\right) = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}} \cdot \frac{e^{-z}}{1+e^{-z}} = \sigma(z)(1-\sigma(z)) \quad \blacksquare$$

### Step 5: Chain Rule Through the Layer
$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial a} \cdot \frac{\partial a}{\partial z} = (a-y) \cdot \sigma'(z)$$

$$\frac{\partial L}{\partial w_i} = \frac{\partial L}{\partial z} \cdot \frac{\partial z}{\partial w_i} = \delta \cdot x_i \quad \text{where } \delta = (a-y)\sigma'(z)$$

$$\frac{\partial L}{\partial b} = \delta$$

### Step 6: Multi-Layer (Layer $l$)
$$\delta^{(l)} = \left(\mathbf{W}^{(l+1)T} \delta^{(l+1)}\right) \odot \sigma'(\mathbf{z}^{(l)})$$
$$\frac{\partial L}{\partial \mathbf{W}^{(l)}} = \delta^{(l)} \cdot \mathbf{a}^{(l-1)T}$$

**This recursion propagates gradients BACKWARD through the network — hence "backpropagation"!**

### Step 7: Gradient Descent Update
$$\mathbf{W}^{(l)} \leftarrow \mathbf{W}^{(l)} - \eta \frac{\partial L}{\partial \mathbf{W}^{(l)}}$$

**Convergence:** For convex $L$ with Lipschitz gradient ($\|\nabla L(x) - \nabla L(y)\| \leq L\|x-y\|$):
$$L(w_T) - L(w^*) \leq \frac{\|w_0 - w^*\|^2}{2\eta T}$$

Convergence rate: $O(1/T)$ for learning rate $\eta \leq 1/L$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_moons, make_circles, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True

np.random.seed(42)
print('All libraries loaded successfully!')

---

## Part 1: The Perceptron — Where It All Began

### 1.1 Mathematical Model

The **perceptron** (Rosenblatt, 1958) is the simplest neural unit. Given input vector $\mathbf{x} \in \mathbb{R}^n$:

$$z = \mathbf{w}^\top \mathbf{x} + b = \sum_{i=1}^{n} w_i x_i + b$$

$$\hat{y} = \sigma(z) = \begin{cases} 1 & \text{if } z \geq 0 \\ 0 & \text{if } z < 0 \end{cases}$$

where:
- $\mathbf{w} \in \mathbb{R}^n$ = weight vector
- $b \in \mathbb{R}$ = bias term
- $\sigma$ = activation function (step function for the classic perceptron)

### 1.2 Geometric Interpretation

The equation $\mathbf{w}^\top \mathbf{x} + b = 0$ defines a **hyperplane** that separates the input space. In 2D:

$$w_1 x_1 + w_2 x_2 + b = 0 \quad \Longrightarrow \quad x_2 = -\frac{w_1}{w_2} x_1 - \frac{b}{w_2}$$

The weight vector $\mathbf{w}$ is **normal** to the decision boundary, and the bias $b$ shifts it from the origin.

In [ ]:
class Perceptron:
    """Classic Rosenblatt perceptron with step activation."""

    def __init__(self, n_features, lr=0.1):
        self.w = np.zeros(n_features)
        self.b = 0.0
        self.lr = lr

    def predict(self, X):
        return (X @ self.w + self.b >= 0).astype(int)

    def fit(self, X, y, epochs=100):
        history = []
        for epoch in range(epochs):
            errors = 0
            for xi, yi in zip(X, y):
                y_hat = int(xi @ self.w + self.b >= 0)
                error = yi - y_hat
                if error != 0:
                    self.w += self.lr * error * xi
                    self.b += self.lr * error
                    errors += 1
            history.append(errors)
            if errors == 0:
                print(f'Converged at epoch {epoch + 1}')
                break
        return history


# Generate linearly separable data
np.random.seed(42)
X_class0 = np.random.randn(50, 2) + np.array([2, 2])
X_class1 = np.random.randn(50, 2) + np.array([-2, -2])
X_data = np.vstack([X_class0, X_class1])
y_data = np.array([0] * 50 + [1] * 50)

p = Perceptron(n_features=2, lr=0.1)
history = p.fit(X_data, y_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Decision boundary
ax = axes[0]
x_min, x_max = X_data[:, 0].min() - 1, X_data[:, 0].max() + 1
y_min, y_max = X_data[:, 1].min() - 1, X_data[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
Z = p.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF9999', '#9999FF']))
ax.scatter(X_class0[:, 0], X_class0[:, 1], c='red', label='Class 0', edgecolors='k')
ax.scatter(X_class1[:, 0], X_class1[:, 1], c='blue', label='Class 1', edgecolors='k')
if abs(p.w[1]) > 1e-8:
    x_line = np.linspace(x_min, x_max, 100)
    y_line = -(p.w[0] * x_line + p.b) / p.w[1]
    ax.plot(x_line, y_line, 'k-', linewidth=2, label='Decision boundary')
    ax.arrow(0, 0, p.w[0], p.w[1], head_width=0.2, head_length=0.1,
             fc='green', ec='green', linewidth=2)
    ax.annotate('w (normal)', xy=(p.w[0], p.w[1]), fontsize=11, color='green')
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('Perceptron Decision Boundary')
ax.legend()

axes[1].plot(history, 'b-o', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Misclassifications')
axes[1].set_title('Perceptron Convergence')

plt.tight_layout()
plt.show()

### 1.3 The XOR Problem — Why We Need Depth

The XOR function cannot be separated by a single hyperplane:

| $x_1$ | $x_2$ | $x_1 \oplus x_2$ |
|-------|-------|-------------------|
| 0     | 0     | 0                 |
| 0     | 1     | 1                 |
| 1     | 0     | 1                 |
| 1     | 1     | 0                 |

**Minsky & Papert (1969)** proved this formally. The solution: **stack layers** to create nonlinear decision boundaries.

In [ ]:
# Demonstrate XOR failure
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_xor = np.array([0, 1, 1, 0])

p_xor = Perceptron(n_features=2, lr=0.1)
history_xor = p_xor.fit(X_xor, y_xor, epochs=1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
Z = p_xor.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF9999', '#9999FF']))
for i, (xi, yi) in enumerate(zip(X_xor, y_xor)):
    color = 'red' if yi == 0 else 'blue'
    ax.scatter(xi[0], xi[1], c=color, s=200, edgecolors='k', linewidths=2, zorder=5)
    ax.annotate(f'({xi[0]},{xi[1]})→{yi}', xy=(xi[0] + 0.05, xi[1] + 0.1), fontsize=11)
ax.set_title('XOR: No Linear Boundary Exists', fontsize=14)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')

axes[1].plot(history_xor[-50:], 'r-o', markersize=3)
axes[1].set_xlabel('Epoch (last 50)')
axes[1].set_ylabel('Misclassifications')
axes[1].set_title('XOR: Perceptron Never Converges')
axes[1].axhline(y=0, color='green', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
print(f'Final misclassifications: {history_xor[-1]} (perceptron cannot solve XOR!)')

---

## Part 2: Activation Functions — The Source of Nonlinearity

Without nonlinear activations, any depth of linear layers collapses to a single linear transformation:

$$\mathbf{W}_2(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2 = \underbrace{(\mathbf{W}_2 \mathbf{W}_1)}_{\mathbf{W}'} \mathbf{x} + \underbrace{(\mathbf{W}_2 \mathbf{b}_1 + \mathbf{b}_2)}_{\mathbf{b}'}$$

### 2.1 Sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Derivative:**

$$\sigma'(z) = \sigma(z)(1 - \sigma(z))$$

**Proof:**
$$\sigma'(z) = \frac{e^{-z}}{(1+e^{-z})^2} = \frac{1}{1+e^{-z}} \cdot \frac{e^{-z}}{1+e^{-z}} = \sigma(z) \cdot \frac{1+e^{-z}-1}{1+e^{-z}} = \sigma(z)(1-\sigma(z))$$

- Range: $(0, 1)$
- Max derivative: $\sigma'(0) = 0.25$ → **vanishing gradient problem**

### 2.2 Hyperbolic Tangent (tanh)

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} = 2\sigma(2z) - 1$$

**Derivative:**

$$\tanh'(z) = 1 - \tanh^2(z)$$

- Range: $(-1, 1)$ — zero-centered, better for gradient flow
- Max derivative: $\tanh'(0) = 1$

### 2.3 ReLU (Rectified Linear Unit)

$$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & \text{if } z > 0 \\ 0 & \text{if } z \leq 0 \end{cases}$$

**Derivative:**

$$\text{ReLU}'(z) = \begin{cases} 1 & \text{if } z > 0 \\ 0 & \text{if } z < 0 \end{cases}$$

- No vanishing gradient for $z > 0$
- Sparse activation → computational efficiency
- **Dying ReLU problem**: neurons with $z < 0$ forever have zero gradient

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z) ** 2

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)


z = np.linspace(-6, 6, 500)

activations = [
    ('Sigmoid', sigmoid, sigmoid_derivative, '#e74c3c'),
    ('Tanh', tanh, tanh_derivative, '#2ecc71'),
    ('ReLU', relu, relu_derivative, '#3498db'),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for i, (name, fn, dfn, color) in enumerate(activations):
    # Function plot
    axes[0, i].plot(z, fn(z), color=color, linewidth=2.5)
    axes[0, i].axhline(y=0, color='gray', linewidth=0.5)
    axes[0, i].axvline(x=0, color='gray', linewidth=0.5)
    axes[0, i].set_title(f'{name}: $\\sigma(z)$', fontsize=14)
    axes[0, i].set_xlabel('z')
    axes[0, i].set_ylabel('$\\sigma(z)$')
    axes[0, i].set_ylim(-1.5, 1.5) if name != 'ReLU' else axes[0, i].set_ylim(-0.5, 6)

    # Derivative plot
    axes[1, i].plot(z, dfn(z), color=color, linewidth=2.5, linestyle='--')
    axes[1, i].axhline(y=0, color='gray', linewidth=0.5)
    axes[1, i].axvline(x=0, color='gray', linewidth=0.5)
    axes[1, i].set_title(f'{name}: $\\sigma\'(z)$', fontsize=14)
    axes[1, i].set_xlabel('z')
    axes[1, i].set_ylabel("$\\sigma'(z)$")
    axes[1, i].fill_between(z, dfn(z), alpha=0.15, color=color)

plt.suptitle('Activation Functions and Their Derivatives', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key observations:')
print(f'  Sigmoid max derivative: {sigmoid_derivative(0):.4f} (at z=0) — vanishing gradients!')
print(f'  Tanh max derivative:    {tanh_derivative(0):.4f} (at z=0) — better but still saturates')
print(f'  ReLU derivative:        {relu_derivative(1.0):.4f} (for z>0) — constant gradient!')

### 2.4 Vanishing Gradient Visualization

For a deep network with $L$ sigmoid layers, the gradient through all layers is:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \prod_{l=2}^{L} \sigma'(z_l) \cdot \mathbf{W}_l \cdot \sigma'(z_1)$$

Since $\max(\sigma'(z)) = 0.25$, the gradient shrinks as $\sim 0.25^L$:

In [ ]:
layers = np.arange(1, 21)
grad_sigmoid = 0.25 ** layers
grad_tanh = 1.0 ** layers  # best case
grad_relu = 1.0 ** layers

plt.figure(figsize=(10, 5))
plt.semilogy(layers, grad_sigmoid, 'r-o', label='Sigmoid (worst case: $0.25^L$)', markersize=5)
plt.semilogy(layers, grad_relu, 'b-s', label='ReLU (best case: $1.0^L$)', markersize=5)
plt.xlabel('Number of Layers $L$', fontsize=13)
plt.ylabel('Gradient Magnitude (log scale)', fontsize=13)
plt.title('Why ReLU Enables Deep Networks', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'After 10 sigmoid layers, gradient shrinks to: {0.25**10:.2e}')
print(f'After 20 sigmoid layers, gradient shrinks to: {0.25**20:.2e}')

---

## Part 3: The Forward Pass — Computing Predictions

### 3.1 Network Architecture

Consider a fully-connected network with $L$ layers. For layer $l \in \{1, \dots, L\}$:

$$\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$$
$$\mathbf{a}^{[l]} = g^{[l]}(\mathbf{z}^{[l]})$$

where:
- $\mathbf{a}^{[0]} = \mathbf{x}$ (input)
- $\mathbf{W}^{[l]} \in \mathbb{R}^{n_l \times n_{l-1}}$ = weight matrix for layer $l$
- $\mathbf{b}^{[l]} \in \mathbb{R}^{n_l}$ = bias vector for layer $l$
- $g^{[l]}$ = activation function for layer $l$
- $n_l$ = number of neurons in layer $l$

### 3.2 Matrix Dimensions

For a batch of $m$ samples, if layer $l$ has $n_l$ neurons:

| Quantity | Dimensions |
|----------|------------|
| $\mathbf{A}^{[l-1]}$ | $n_{l-1} \times m$ |
| $\mathbf{W}^{[l]}$ | $n_l \times n_{l-1}$ |
| $\mathbf{b}^{[l]}$ | $n_l \times 1$ (broadcast) |
| $\mathbf{Z}^{[l]}$ | $n_l \times m$ |
| $\mathbf{A}^{[l]}$ | $n_l \times m$ |

In [ ]:
def visualize_forward_pass():
    """Visualize data flowing through a small network."""
    np.random.seed(0)
    x = np.array([[0.5], [0.8]])

    W1 = np.array([[0.2, -0.5], [0.7, 0.1], [-0.3, 0.9]])
    b1 = np.array([[0.1], [-0.2], [0.3]])
    z1 = W1 @ x + b1
    a1 = relu(z1)

    W2 = np.array([[0.4, -0.6, 0.2]])
    b2 = np.array([[0.1]])
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))

    data_arrays = [
        (x, 'Input\n$\mathbf{x}$'),
        (z1, 'Pre-activation\n$\mathbf{z}^{[1]} = W^{[1]}x + b^{[1]}$'),
        (a1, 'Post-activation\n$\mathbf{a}^{[1]} = \mathrm{ReLU}(z^{[1]})$'),
        (z2, 'Pre-activation\n$z^{[2]} = W^{[2]}a^{[1]} + b^{[2]}$'),
        (a2, 'Output\n$\hat{y} = \sigma(z^{[2]})$'),
    ]

    for ax, (arr, title) in zip(axes, data_arrays):
        im = ax.imshow(arr, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                ax.text(j, i, f'{arr[i,j]:.3f}', ha='center', va='center',
                        fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=10)
        ax.set_xticks([])
        ax.set_yticks(range(arr.shape[0]))

    plt.suptitle('Forward Pass: Data Flow Through the Network', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print('Forward pass computation:')
    print(f'  Input x = {x.flatten()}')
    print(f'  z1 = W1 @ x + b1 = {z1.flatten()}')
    print(f'  a1 = ReLU(z1)     = {a1.flatten()}')
    print(f'  z2 = W2 @ a1 + b2 = {z2.flatten()}')
    print(f'  ŷ  = sigmoid(z2)  = {a2.flatten()}')

visualize_forward_pass()

---

## Part 4: Backpropagation — The Full Derivation

### 4.1 Loss Function

For binary classification with sigmoid output, we use the **binary cross-entropy** loss:

$$\mathcal{L}(y, \hat{y}) = -\left[ y \log(\hat{y}) + (1 - y) \log(1 - \hat{y}) \right]$$

For $m$ samples, the **cost function**:

$$J = \frac{1}{m} \sum_{i=1}^{m} \mathcal{L}(y^{(i)}, \hat{y}^{(i)})$$

### 4.2 Backpropagation via Chain Rule

We need $\frac{\partial J}{\partial \mathbf{W}^{[l]}}$ and $\frac{\partial J}{\partial \mathbf{b}^{[l]}}$ for each layer.

**Step 1: Output layer gradient ($l = L$, sigmoid activation)**

$$\frac{\partial \mathcal{L}}{\partial \hat{y}} = -\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}}$$

$$\frac{\partial \hat{y}}{\partial z^{[L]}} = \hat{y}(1 - \hat{y})$$

Combining via chain rule:

$$\delta^{[L]} = \frac{\partial \mathcal{L}}{\partial z^{[L]}} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z^{[L]}} = \left(-\frac{y}{\hat{y}} + \frac{1-y}{1-\hat{y}}\right) \cdot \hat{y}(1-\hat{y})$$

Simplifying:

$$\boxed{\delta^{[L]} = \hat{y} - y = \mathbf{a}^{[L]} - \mathbf{y}}$$

**Step 2: Hidden layer gradient ($l < L$)**

Using the chain rule backward through layer $l$:

$$\delta^{[l]} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}^{[l]}} = \underbrace{\left(\mathbf{W}^{[l+1]}\right)^\top \delta^{[l+1]}}_{\text{error propagated back}} \odot \underbrace{g'^{[l]}(\mathbf{z}^{[l]})}_{\text{local gradient}}$$

where $\odot$ is element-wise multiplication.

**Step 3: Parameter gradients**

Since $\mathbf{z}^{[l]} = \mathbf{W}^{[l]} \mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$:

$$\boxed{\frac{\partial J}{\partial \mathbf{W}^{[l]}} = \frac{1}{m} \delta^{[l]} \left(\mathbf{a}^{[l-1]}\right)^\top}$$

$$\boxed{\frac{\partial J}{\partial \mathbf{b}^{[l]}} = \frac{1}{m} \sum_{i=1}^{m} \delta^{[l](i)}}$$

### 4.3 The Complete Chain Rule Unrolled

For a 3-layer network, the gradient of $J$ w.r.t. $\mathbf{W}^{[1]}$ is:

$$\frac{\partial J}{\partial \mathbf{W}^{[1]}} = \underbrace{\frac{\partial J}{\partial \mathbf{a}^{[3]}}}_{\text{loss grad}} \cdot \underbrace{\frac{\partial \mathbf{a}^{[3]}}{\partial \mathbf{z}^{[3]}}}_{g'^{[3]}} \cdot \underbrace{\frac{\partial \mathbf{z}^{[3]}}{\partial \mathbf{a}^{[2]}}}_{\mathbf{W}^{[3]}} \cdot \underbrace{\frac{\partial \mathbf{a}^{[2]}}{\partial \mathbf{z}^{[2]}}}_{g'^{[2]}} \cdot \underbrace{\frac{\partial \mathbf{z}^{[2]}}{\partial \mathbf{a}^{[1]}}}_{\mathbf{W}^{[2]}} \cdot \underbrace{\frac{\partial \mathbf{a}^{[1]}}{\partial \mathbf{z}^{[1]}}}_{g'^{[1]}} \cdot \underbrace{\frac{\partial \mathbf{z}^{[1]}}{\partial \mathbf{W}^{[1]}}}_{\mathbf{a}^{[0]}}$$

In [ ]:
def numerical_gradient_check():
    """
    Verify backpropagation against numerical gradients.
    This is the gold standard for debugging neural network implementations.
    """
    np.random.seed(42)
    x = np.random.randn(2, 1)
    y = np.array([[1.0]])

    W1 = np.random.randn(3, 2) * 0.5
    b1 = np.zeros((3, 1))
    W2 = np.random.randn(1, 3) * 0.5
    b2 = np.zeros((1, 1))

    # Analytical forward + backward
    z1 = W1 @ x + b1
    a1 = relu(z1)
    z2 = W2 @ a1 + b2
    a2 = sigmoid(z2)
    loss = -(y * np.log(a2 + 1e-8) + (1 - y) * np.log(1 - a2 + 1e-8))

    # Backprop
    dz2 = a2 - y
    dW2 = dz2 @ a1.T
    db2 = dz2
    da1 = W2.T @ dz2
    dz1 = da1 * relu_derivative(z1)
    dW1 = dz1 @ x.T
    db1 = dz1

    # Numerical gradient for W1
    epsilon = 1e-7
    dW1_num = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            W1_plus = W1.copy()
            W1_plus[i, j] += epsilon
            z1_p = W1_plus @ x + b1
            a1_p = relu(z1_p)
            z2_p = W2 @ a1_p + b2
            a2_p = sigmoid(z2_p)
            loss_plus = -(y * np.log(a2_p + 1e-8) + (1 - y) * np.log(1 - a2_p + 1e-8))

            W1_minus = W1.copy()
            W1_minus[i, j] -= epsilon
            z1_m = W1_minus @ x + b1
            a1_m = relu(z1_m)
            z2_m = W2 @ a1_m + b2
            a2_m = sigmoid(z2_m)
            loss_minus = -(y * np.log(a2_m + 1e-8) + (1 - y) * np.log(1 - a2_m + 1e-8))

            dW1_num[i, j] = (loss_plus - loss_minus) / (2 * epsilon)

    rel_error = np.linalg.norm(dW1 - dW1_num) / (np.linalg.norm(dW1) + np.linalg.norm(dW1_num) + 1e-8)

    print('=== Gradient Check ===')
    print(f'Analytical dW1:\n{dW1}')
    print(f'\nNumerical dW1:\n{dW1_num}')
    print(f'\nRelative error: {rel_error:.2e}')
    print(f'Gradient check {"PASSED ✓" if rel_error < 1e-5 else "FAILED ✗"}')

numerical_gradient_check()

---

## Part 5: Gradient Descent — Optimization

### 5.1 The Update Rule

**Vanilla gradient descent** updates parameters in the direction of steepest descent:

$$\mathbf{W}^{[l]} \leftarrow \mathbf{W}^{[l]} - \alpha \frac{\partial J}{\partial \mathbf{W}^{[l]}}$$

$$\mathbf{b}^{[l]} \leftarrow \mathbf{b}^{[l]} - \alpha \frac{\partial J}{\partial \mathbf{b}^{[l]}}$$

where $\alpha > 0$ is the **learning rate**.

### 5.2 Variants

| Variant | Update Rule | Key Property |
|---------|------------|---------------|
| Batch GD | $\theta \leftarrow \theta - \alpha \nabla_{\theta} J(\theta)$ | Uses all samples |
| Stochastic GD | $\theta \leftarrow \theta - \alpha \nabla_{\theta} J(\theta; x^{(i)}, y^{(i)})$ | Uses one sample |
| Mini-batch GD | $\theta \leftarrow \theta - \alpha \nabla_{\theta} J(\theta; \mathcal{B})$ | Uses batch $\mathcal{B}$ |

### 5.3 Learning Rate Effect

The learning rate $\alpha$ controls the step size. Too large → divergence. Too small → slow convergence.

In [ ]:
def visualize_gradient_descent_2d():
    """Visualize gradient descent on a 2D loss surface."""
    def loss_fn(w1, w2):
        return (w1 - 1)**2 + 5 * (w2 + 0.5)**2 + 0.5 * w1 * w2

    def grad_fn(w1, w2):
        dw1 = 2 * (w1 - 1) + 0.5 * w2
        dw2 = 10 * (w2 + 0.5) + 0.5 * w1
        return dw1, dw2

    learning_rates = [0.01, 0.05, 0.15]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    w1_range = np.linspace(-3, 4, 200)
    w2_range = np.linspace(-3, 3, 200)
    W1_grid, W2_grid = np.meshgrid(w1_range, w2_range)
    L_grid = loss_fn(W1_grid, W2_grid)

    for ax, lr in zip(axes, learning_rates):
        ax.contour(W1_grid, W2_grid, L_grid, levels=30, cmap='viridis', alpha=0.7)

        w = np.array([-2.5, 2.5])
        path = [w.copy()]
        for _ in range(50):
            g1, g2 = grad_fn(w[0], w[1])
            w[0] -= lr * g1
            w[1] -= lr * g2
            path.append(w.copy())
            if loss_fn(w[0], w[1]) > 1e6:
                break

        path = np.array(path)
        ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=6, linewidth=1.5)
        ax.plot(path[0, 0], path[0, 1], 'go', markersize=12, label='Start')
        ax.plot(path[-1, 0], path[-1, 1], 'r*', markersize=15, label='End')
        ax.plot(1, -0.5, 'k*', markersize=15, label='Optimum')
        ax.set_title(f'$\\alpha = {lr}$ ({len(path)} steps)', fontsize=13)
        ax.set_xlabel('$w_1$')
        ax.set_ylabel('$w_2$')
        ax.legend(fontsize=9)
        ax.set_xlim(-3, 4)
        ax.set_ylim(-3, 3)

    plt.suptitle('Gradient Descent: Effect of Learning Rate', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_gradient_descent_2d()

---

## Part 6: Build a Complete Neural Network from Scratch

We now implement a fully-connected neural network class using only NumPy. This network will:
1. Support arbitrary layer sizes
2. Use ReLU for hidden layers and sigmoid for the output
3. Implement full backpropagation
4. Train on image data

### 6.1 Weight Initialization

We use **He initialization** for ReLU layers:

$$\mathbf{W}^{[l]} \sim \mathcal{N}\left(0, \sqrt{\frac{2}{n_{l-1}}}\right)$$

This ensures the variance of activations stays roughly constant across layers.

In [ ]:
class NeuralNetwork:
    """Multi-layer neural network built from scratch with NumPy."""

    def __init__(self, layer_dims):
        """
        layer_dims: list of layer sizes, e.g. [784, 128, 64, 10]
        """
        self.L = len(layer_dims) - 1  # number of layers (excluding input)
        self.params = {}
        self.layer_dims = layer_dims

        for l in range(1, self.L + 1):
            # He initialization
            self.params[f'W{l}'] = np.random.randn(
                layer_dims[l], layer_dims[l - 1]
            ) * np.sqrt(2.0 / layer_dims[l - 1])
            self.params[f'b{l}'] = np.zeros((layer_dims[l], 1))

    def forward(self, X):
        """Forward pass. Returns output and cache for backprop."""
        cache = {'A0': X}
        A = X

        # Hidden layers: ReLU
        for l in range(1, self.L):
            Z = self.params[f'W{l}'] @ A + self.params[f'b{l}']
            A = relu(Z)
            cache[f'Z{l}'] = Z
            cache[f'A{l}'] = A

        # Output layer: sigmoid (binary) or softmax (multi-class)
        ZL = self.params[f'W{self.L}'] @ A + self.params[f'b{self.L}']
        if self.layer_dims[-1] == 1:
            AL = sigmoid(ZL)
        else:
            AL = self._softmax(ZL)

        cache[f'Z{self.L}'] = ZL
        cache[f'A{self.L}'] = AL
        return AL, cache

    def _softmax(self, Z):
        exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
        return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

    def compute_loss(self, AL, Y):
        """Cross-entropy loss."""
        m = Y.shape[1]
        if self.layer_dims[-1] == 1:
            cost = -np.sum(Y * np.log(AL + 1e-8) + (1 - Y) * np.log(1 - AL + 1e-8)) / m
        else:
            cost = -np.sum(Y * np.log(AL + 1e-8)) / m
        return float(np.squeeze(cost))

    def backward(self, Y, cache):
        """Backpropagation. Returns gradients for all parameters."""
        grads = {}
        m = Y.shape[1]
        AL = cache[f'A{self.L}']

        # Output layer gradient
        dZ = AL - Y  # works for both sigmoid+BCE and softmax+CE
        grads[f'dW{self.L}'] = (1 / m) * dZ @ cache[f'A{self.L - 1}'].T
        grads[f'db{self.L}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)

        # Hidden layers
        for l in range(self.L - 1, 0, -1):
            dA = self.params[f'W{l + 1}'].T @ dZ
            dZ = dA * relu_derivative(cache[f'Z{l}'])
            grads[f'dW{l}'] = (1 / m) * dZ @ cache[f'A{l - 1}'].T
            grads[f'db{l}'] = (1 / m) * np.sum(dZ, axis=1, keepdims=True)

        return grads

    def update_params(self, grads, lr):
        """Gradient descent update."""
        for l in range(1, self.L + 1):
            self.params[f'W{l}'] -= lr * grads[f'dW{l}']
            self.params[f'b{l}'] -= lr * grads[f'db{l}']

    def train(self, X, Y, epochs=1000, lr=0.01, print_every=100):
        """Full training loop."""
        losses = []
        for epoch in range(epochs):
            AL, cache = self.forward(X)
            loss = self.compute_loss(AL, Y)
            grads = self.backward(Y, cache)
            self.update_params(grads, lr)
            losses.append(loss)
            if (epoch + 1) % print_every == 0:
                acc = self.accuracy(X, Y)
                print(f'Epoch {epoch+1:4d}/{epochs} — Loss: {loss:.4f} — Acc: {acc:.2%}')
        return losses

    def predict(self, X):
        AL, _ = self.forward(X)
        if self.layer_dims[-1] == 1:
            return (AL > 0.5).astype(int)
        else:
            return np.argmax(AL, axis=0)

    def accuracy(self, X, Y):
        preds = self.predict(X)
        if self.layer_dims[-1] == 1:
            return np.mean(preds == Y)
        else:
            return np.mean(preds == np.argmax(Y, axis=0))

print('NeuralNetwork class defined!')

### 6.2 Test on XOR Problem

Our multi-layer network should solve what the perceptron could not:

In [ ]:
X_xor_nn = np.array([[0, 0, 1, 1], [0, 1, 0, 1]], dtype=float)
Y_xor_nn = np.array([[0, 1, 1, 0]], dtype=float)

nn_xor = NeuralNetwork([2, 8, 1])
losses_xor = nn_xor.train(X_xor_nn, Y_xor_nn, epochs=5000, lr=0.5, print_every=1000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_xor, 'b-', linewidth=1)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('XOR Training Loss')

ax = axes[1]
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()].T
probs, _ = nn_xor.forward(grid)
Z = probs.reshape(xx.shape)
ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.8)
ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
for i in range(4):
    color = 'red' if Y_xor_nn[0, i] == 0 else 'blue'
    ax.scatter(X_xor_nn[0, i], X_xor_nn[1, i], c=color, s=200, edgecolors='k',
               linewidths=2, zorder=5)
ax.set_title('XOR: Neural Network Decision Boundary', fontsize=13)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
plt.colorbar(ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.0), ax=ax, label='P(y=1)')

plt.tight_layout()
plt.show()

print('\nPredictions:')
for i in range(4):
    x_in = X_xor_nn[:, i:i+1]
    prob, _ = nn_xor.forward(x_in)
    print(f'  Input ({X_xor_nn[0,i]:.0f}, {X_xor_nn[1,i]:.0f}) → P(y=1) = {prob[0,0]:.4f} → Predicted: {int(prob[0,0] > 0.5)}, True: {int(Y_xor_nn[0,i])}')

### 6.3 Nonlinear Decision Boundaries

Let's train on the **moons** dataset to see how network depth affects the decision boundary:

In [ ]:
X_moons, y_moons = make_moons(n_samples=500, noise=0.2, random_state=42)
X_moons = X_moons.T  # shape (2, 500)
Y_moons = y_moons.reshape(1, -1)

architectures = [
    ([2, 1], 'No hidden layer'),
    ([2, 4, 1], '1 hidden (4 neurons)'),
    ([2, 16, 1], '1 hidden (16 neurons)'),
    ([2, 16, 8, 1], '2 hidden (16, 8)'),
]

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))

for ax, (arch, name) in zip(axes, architectures):
    nn = NeuralNetwork(arch)
    nn.train(X_moons, Y_moons, epochs=3000, lr=0.5, print_every=99999)

    xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    grid = np.c_[xx.ravel(), yy.ravel()].T
    probs, _ = nn.forward(grid)
    Z = probs.reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=50, cmap='RdBu_r', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    ax.scatter(X_moons[0, y_moons == 0], X_moons[1, y_moons == 0],
               c='red', s=15, edgecolors='k', linewidths=0.5)
    ax.scatter(X_moons[0, y_moons == 1], X_moons[1, y_moons == 1],
               c='blue', s=15, edgecolors='k', linewidths=0.5)

    acc = nn.accuracy(X_moons, Y_moons)
    ax.set_title(f'{name}\nAcc: {acc:.1%}', fontsize=11)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Effect of Network Depth on Decision Boundary', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 7: Classify Tiny Images — Putting It All Together

We use the **sklearn digits** dataset (8×8 grayscale images of digits 0–9) to demonstrate our NumPy neural network on real image data.

Each image is a 64-dimensional vector (8×8 pixels flattened). This is our **state representation** — the same concept we will use in Deep RL, where images define the agent's state.

In [ ]:
digits = load_digits()
X_digits = digits.data  # (1797, 64)
y_digits = digits.target  # (1797,)

fig, axes = plt.subplots(2, 10, figsize=(16, 4))
for i in range(20):
    ax = axes[i // 10, i % 10]
    ax.imshow(X_digits[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'{y_digits[i]}', fontsize=11)
    ax.axis('off')
plt.suptitle('Sample 8×8 Digit Images (our tiny image dataset)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Dataset: {X_digits.shape[0]} images, {X_digits.shape[1]} features (8×8 pixels)')
print(f'Classes: {np.unique(y_digits)}')

In [ ]:
# Prepare data
X_train, X_test, y_train, y_test = train_test_split(
    X_digits, y_digits, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).T  # shape (64, n_train)
X_test_scaled = scaler.transform(X_test).T  # shape (64, n_test)

# One-hot encode labels
def one_hot(y, num_classes):
    oh = np.zeros((num_classes, len(y)))
    oh[y, np.arange(len(y))] = 1
    return oh

Y_train = one_hot(y_train, 10)
Y_test = one_hot(y_test, 10)

print(f'Training set: X={X_train_scaled.shape}, Y={Y_train.shape}')
print(f'Test set:     X={X_test_scaled.shape}, Y={Y_test.shape}')

In [ ]:
# Train the network: input(64) -> hidden(128) -> hidden(64) -> output(10)
nn_digits = NeuralNetwork([64, 128, 64, 10])

print('Training neural network on 8×8 digit images...')
print('Architecture: 64 → 128 → 64 → 10')
print('=' * 50)

losses = nn_digits.train(
    X_train_scaled, Y_train,
    epochs=2000, lr=0.1, print_every=200
)

train_acc = nn_digits.accuracy(X_train_scaled, Y_train)
test_acc = nn_digits.accuracy(X_test_scaled, Y_test)
print(f'\nFinal Train Accuracy: {train_acc:.2%}')
print(f'Final Test Accuracy:  {test_acc:.2%}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(losses, 'b-', linewidth=1)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss')

# Sample predictions
ax = axes[1]
preds = nn_digits.predict(X_test_scaled)
n_show = 20
idxs = np.random.choice(len(y_test), n_show, replace=False)
for i, idx in enumerate(idxs[:10]):
    row, col = divmod(i, 5)
    x_pos = col * 1.5
    y_pos = (1 - row) * 1.5
    ax.imshow(X_test[idx].reshape(8, 8), cmap='gray',
              extent=[x_pos, x_pos + 1, y_pos, y_pos + 1])
    color = 'green' if preds[idx] == y_test[idx] else 'red'
    ax.text(x_pos + 0.5, y_pos - 0.15, f'P:{preds[idx]} T:{y_test[idx]}',
            ha='center', fontsize=8, color=color, fontweight='bold')
ax.set_xlim(-0.2, 7.5)
ax.set_ylim(-0.5, 3.2)
ax.set_title('Sample Predictions (P=pred, T=true)')
ax.axis('off')

# Confusion-style accuracy per digit
per_class_acc = []
for d in range(10):
    mask = y_test == d
    if mask.sum() > 0:
        per_class_acc.append(np.mean(preds[mask] == y_test[mask]))
    else:
        per_class_acc.append(0)

axes[2].bar(range(10), per_class_acc, color='steelblue', edgecolor='k')
axes[2].set_xlabel('Digit')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Per-Class Accuracy')
axes[2].set_xticks(range(10))
axes[2].set_ylim(0, 1.05)
for d, acc in enumerate(per_class_acc):
    axes[2].text(d, acc + 0.02, f'{acc:.0%}', ha='center', fontsize=9)

plt.suptitle(f'NumPy Neural Network on Digit Images — Test Accuracy: {test_acc:.1%}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.1 Visualizing What the Network Learns

The first layer weights can be reshaped back to 8×8 to see what patterns each neuron detects:

In [ ]:
W1 = nn_digits.params['W1']  # shape (128, 64)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < W1.shape[0]:
        w = W1[i].reshape(8, 8)
        ax.imshow(w, cmap='RdBu_r', interpolation='nearest')
    ax.axis('off')
    ax.set_title(f'N{i}', fontsize=8)

plt.suptitle('First Layer Weights Visualized as 8×8 Filters\n'
             '(Each neuron learns a different pattern detector)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('These weight patterns are analogous to convolutional filters!')
print('In Module 5.2, we will make this spatial structure explicit with CNNs.')

---

## 🔑 Key Takeaways

| Concept | Key Insight |
|---------|------------|
| **Perceptron** | Linear classifier; can't solve XOR → need nonlinearity |
| **Activation functions** | ReLU preferred: no vanishing gradient, sparse, fast |
| **Forward pass** | $\mathbf{z}^{[l]} = \mathbf{W}^{[l]}\mathbf{a}^{[l-1]} + \mathbf{b}^{[l]}$, $\mathbf{a}^{[l]} = g(\mathbf{z}^{[l]})$ |
| **Backpropagation** | Chain rule backward: $\delta^{[l]} = (\mathbf{W}^{[l+1]})^\top \delta^{[l+1]} \odot g'(\mathbf{z}^{[l]})$ |
| **Gradient descent** | $\theta \leftarrow \theta - \alpha \nabla_\theta J$ |
| **For RL** | Neural networks approximate Q-functions and policies in Deep RL |

### Connection to RL

In Deep RL, neural networks serve two critical roles:
1. **Value networks** approximate $Q(s, a; \theta)$ — the expected return of taking action $a$ in state $s$
2. **Policy networks** approximate $\pi(a|s; \theta)$ — the probability of actions given states

The backpropagation machinery we built here is identical to what trains DQN, A2C, and PPO agents.

---

## ➡️ Next: Module 5.2 — CNN From Scratch

Fully-connected networks treat each pixel independently. **Convolutional Neural Networks** exploit spatial structure in images, directly connecting to the filters from Module 2!